# Impact Split Explainer

This notebook documents the story behind the math in `impact-split`: why it was built, how its formulas evolved, and how those formulas map to what the fitted tree actually shows.

Flow of this notebook:

1. The business paradox that average-purity trees miss
2. The three-act mathematical design (`delta`, gain, dual materiality)
3. A toy scenario with a documented synthetic DGP (planted category interactions, noise, observed outcome)
4. What to inspect in the generated tree and segments
5. How parameter choices change what we see

If you only need a high-level summary, use `README.md`. This notebook is the deeper walkthrough.

## 1) Problem Framing: Why This Needed a New Tree

Most classic trees (CART/C4.5 family) were designed to optimize variance and average purity. That is mathematically coherent, but for additive business KPIs it can be misleading.

A simple paradox:

- Segment A: 2 users x -5,000 average = -10,000 total
- Segment B: 10,000 users x -40 average = -400,000 total

Average-first trees can chase Segment A because it is "purer" by rate, while business decisions should often prioritize Segment B because it dominates total impact.

`impact-split` was created to optimize for additive totals directly, not average purity.

> Guiding question: which segments hold the most positive or negative mathematical material in absolute business terms?

## 2) The Three-Act Mathematical Design

The formulas below are the core design choices behind `impact-split`.

### Act I: Local Sieve (`delta`)

**Problem:** binary splits force categories into good/bad buckets and hide the baseline. We need Positive / Neutral / Negative routing.

At each node:

$$
V_{node} = \sum_{j=1}^{N_{node}} |y_j|
$$

$$
\delta = V_{node} \times \text{delta\_pct}
$$

For each category in feature $X_i$:

$$
S_{cat} = \sum_{y \in cat} y
$$

Routing rule:

- Positive if $S_{cat} > \delta$
- Negative if $S_{cat} < -\delta$
- Neutral otherwise

Why this matters: the neutral band auto-scales with local node volume, so the tree behaves like a zoom lens. Large nodes ignore tiny swings; deeper nodes can detect finer impacts.

**Implementation detail:** current splitter logic applies the scaled rule $\delta = V_{node} \times \text{delta\_pct}$ at every depth, including the root.

### Act II: Gain Metric (Category-Averaged Impact Divergence)

**Problem:** after ternary routing, we need a split score that rewards material impact but resists high-cardinality overfit.

Final score:

$$
Gain(X_i) = \frac{|S_P|}{k_P} + \frac{|S_N|}{k_N}
$$

Where $S_P, S_N$ are outer-branch sums and $k_P, k_N$ are category counts per outer branch.

Why this matters: the score balances volume and density, favoring features that isolate large positive/negative totals in fewer actionable categories.

### Act III: Global Kill Switch (Dual Materiality)

**Problem:** as $\delta$ shrinks down the tree, tiny local noise can eventually look important unless we stop by global business relevance.

Define global positive/negative maxima once:

$$
V_{global\_P} = \sum_{y_i > 0} y_i,
\quad
V_{global\_N} = \sum_{y_i < 0} |y_i|
$$

At each node:

$$
S_{node\_P} = \sum_{y_i \in node, y_i > 0} y_i,
\quad
S_{node\_N} = \sum_{y_i \in node, y_i < 0} |y_i|
$$

Stop if both sides are immaterial:

$$
\left(\frac{S_{node\_P}}{V_{global\_P}} \le \theta_{stop}\right)
\quad \text{AND} \quad
\left(\frac{S_{node\_N}}{V_{global\_N}} \le \theta_{stop}\right)
$$

with $\theta_{stop} = \text{min\_global\_impact\_pct}$.

Why this matters: positive mass is graded against the global positive pool, and negative mass against the global negative pool, avoiding net-sum distortions.

Parameter mapping used later in code:

- `delta_pct` controls local sieve width at each node
- `min_global_impact_pct` sets the dual materiality threshold

In [ ]:
import numpy as np
import pandas as pd

from impact_split import ImpactSplitter

## 3) Synthetic data (DGP)

We simulate a **documented** data-generating process so we can compare *what we planted* with *what the tree finds autonomously*:

- **`y_expected`** — row-level contribution from **explicit categorical rules** (two-way and three-way combinations of region, channel, and product). There is **no** separate marginal effect (e.g. region-only bias): the only systematic signal lives in these interaction masks.
- **`epsilon`** — iid noise around each row.

We **fit the tree on `y_observed = y_expected + epsilon`** only (one observed outcome per row, like real data).

The notebook lists each planted rule in `planted_rules_df` with row counts and totals on each rule’s **mask** before fitting. Rule masks are **pairwise disjoint**, so each row receives at most one planted increment and slice totals match `n × planted_increment` for that rule.

`ImpactSplitter` expects **integer label-encoded** features: we build `X_np` and keep `category_maps` so you can decode node labels `f0`, `f1`, `f2` in the plot (`f0` = region, `f1` = channel, `f2` = product).

In [ ]:
# Build a toy scenario for illustration (not a claim about real data)
rng = np.random.default_rng(42)
n = 5000

regions = np.array(["NCR", "Luzon", "Visayas", "Mindanao"])
channels = np.array(["Direct", "Partner", "Online"])
products = np.array(["A", "B", "C"])

X_df = pd.DataFrame(
    {
        "region": rng.choice(regions, size=n, p=[0.35, 0.3, 0.2, 0.15]),
        "channel": rng.choice(channels, size=n, p=[0.25, 0.35, 0.4]),
        "product": rng.choice(products, size=n, p=[0.4, 0.35, 0.25]),
    }
)

# Planted rules: only category *combinations* drive the target (no region-only bias).
# Masks are pairwise disjoint so each row gets at most one structural increment.
rule_specs = [
    ("NCR × Direct", (X_df["region"] == "NCR") & (X_df["channel"] == "Direct"), 120.0),
    ("Mindanao × Partner × {A,B}", (X_df["region"] == "Mindanao") & (X_df["channel"] == "Partner") & (X_df["product"].isin(["A", "B"])), -95.0),
    ("Luzon × Online × C", (X_df["region"] == "Luzon") & (X_df["channel"] == "Online") & (X_df["product"] == "C"), 35.0),
    ("Luzon × Partner", (X_df["region"] == "Luzon") & (X_df["channel"] == "Partner"), 60.0),
    ("Visayas × Online", (X_df["region"] == "Visayas") & (X_df["channel"] == "Online"), -45.0),
    ("Luzon × Online × A (3-way)", (X_df["region"] == "Luzon") & (X_df["channel"] == "Online") & (X_df["product"] == "A"), 50.0),
]

y_expected = np.zeros(n, dtype=np.float64)
for _label, mask, effect in rule_specs:
    y_expected += np.where(mask, effect, 0.0)

# Noise: tuned so planted structure remains visible under default splitter settings.
epsilon = rng.normal(loc=0, scale=22, size=n)
y_observed = y_expected + epsilon

region_codes = pd.Categorical(X_df["region"], categories=regions).codes
channel_codes = pd.Categorical(X_df["channel"], categories=channels).codes
product_codes = pd.Categorical(X_df["product"], categories=products).codes
X_np = np.column_stack([region_codes, channel_codes, product_codes]).astype(np.int64)

category_maps = {
    0: {i: lab for i, lab in enumerate(regions)},
    1: {i: lab for i, lab in enumerate(channels)},
    2: {i: lab for i, lab in enumerate(products)},
}
feature_names = ["region", "channel", "product"]

# Ground truth: one row per planted rule (before fitting the tree).
planted_rows = []
for label, mask, effect in rule_specs:
    m = mask.to_numpy() if hasattr(mask, "to_numpy") else np.asarray(mask)
    planted_rows.append(
        {
            "rule": label,
            "planted_increment": effect,
            "n": int(m.sum()),
            "mean_y_on_slice": float(y_expected[m].mean()),
            "sum_y_expected": float(y_expected[m].sum()),
            "sum_y_observed": float(y_observed[m].sum()),
        }
    )
planted_rules_df = pd.DataFrame(planted_rows)
planted_rules_df["abs_sum_expected"] = planted_rules_df["sum_y_expected"].abs()
# Rows can match multiple rules; sums on a mask are totals of y_expected there (combined effects).
planted_rules_df.sort_values("abs_sum_expected", ascending=False)

In [ ]:
# 4) Fit one baseline configuration for exploration (observed outcome only)
y_fit = y_observed.astype(np.float64, copy=False)

model = ImpactSplitter(
    delta_pct=0.05,
    min_global_impact_pct=0.01,
    max_depth=4,
)

model.fit(X_np, y_fit)

In [ ]:
# Inspect the split structure produced by the baseline run.
model.plot_tree(figsize=(16, 10))

In [ ]:
# Inspect top segments and compare them with the planted rules (ground truth before fit).
segments = model.get_impact_segments()

print(
    "Global sum(y_expected) =",
    float(y_expected.sum()),
    "| Global sum(y_observed) =",
    float(y_observed.sum()),
)
print("\nPlanted rules (structural totals on each mask, sorted by |sum_y_expected|):")
print(
    planted_rules_df.sort_values("abs_sum_expected", ascending=False)[
        ["rule", "planted_increment", "n", "mean_y_on_slice", "sum_y_expected", "sum_y_observed"]
    ].to_string(index=False)
)

top_rules = planted_rules_df.sort_values("abs_sum_expected", ascending=False).head(3)["rule"].tolist()
print("\nLargest planted rules by |structural total| (for eyeballing the tree):", top_rules)
print(
    "\nTop segments by |total_sum| use paths f0=region, f1=channel, f2=product "
    "(positive/negative/neutral = category routing at each split)."
)

seg_by_abs = segments.assign(_abs=np.abs(segments["total_sum"])).sort_values(
    "_abs", ascending=False
).drop(columns=["_abs"])
seg_by_abs.head(15)

### Reading the segments

The tree is fit on **`y_observed`**, so leaf `total_sum` values are **not** numerically equal to sums of `y_expected` on the same hand-built mask (iid **noise** shifts row-level outcomes). You should still see **large-magnitude** leaves align with the **planted interaction rules** in `planted_rules_df`, because those rules dominate pooled category sums at positive/negative branches. Decode `f0` / `f1` / `f2` in `plot_tree` with `category_maps` (`f0` = region, `f1` = channel, `f2` = product).

## 5) Parameter Exploration

This section is intentionally exploratory.

Rather than assume one "correct" setting, compare how the output changes when we vary:

- `delta_pct` (local sieve width)
- `min_global_impact_pct` (global materiality stop)

Suggested reading questions:

- Which segments stay stable across runs?
- Which segments appear only under more aggressive settings?
- How does tree complexity change?

In [ ]:
# Run a small comparison grid and review differences in top segments.
for delta_pct, min_global_impact_pct in [(0.03, 0.01), (0.05, 0.01), (0.08, 0.02)]:
    m = ImpactSplitter(
        delta_pct=delta_pct,
        min_global_impact_pct=min_global_impact_pct,
        max_depth=4,
    )
    m.fit(X_np, y_fit)
    top = m.get_impact_segments().head(3)
    print(f"delta_pct={delta_pct}, min_global_impact_pct={min_global_impact_pct}")
    print(top[["path", "total_sum"]] if {"path", "total_sum"}.issubset(top.columns) else top)
    print("-" * 80)